# Clase 13: Regresión

**MDS7202: Laboratorio de Programación Científica para Ciencia de Datos**

**Profesor: Diego Cortez**


## Objetivos de la Clase

Conceptual
- Comprender lo que es aprendizaje supervisado ✅
- Framework general: objetivos y métricas ✅
- Overfitting y regularización ✅
- Train test split
- El problema de la regresión
- Métricas de regresión
  
Aplicación
- Regresión lineal
- Ridge regression
- Trade-off sesgo-varianza
- Pipeline predictivo
- MLFlow

---

## Machine learning en general

<br>

<div align='center'>
<img src="https://github.com/MDS7202/MDS7202/blob/main/recursos/2023-01/18-Aprendizaje-Supervisado-I/machine_learning.png?raw=true" alt="Panorama General ML: Clasificación supervisada, No supervisada y Aprendizaje Reforzado binario" width=700/>
</div>


En general, machine learning es el conjunto de técnicas en que una máquina o algoritmo puede ***aprender*** un patrón en base a datos. Esto ya lo hemos visto aplicado en 2 grupos de prácticas:
- **Preprocesamiento y Feature engineering.** Procesadores necesitan 'aprender' qué operaciones realizar a una columna para preprocesarla. Ej., Mínimos y máximos, media, cantidad de categorías, etc.
- **Aprendizaje no supervisado.** Un algoritmo aprende agrupaciones, relaciones entre variables, transformaciones que revelan patrones subyacentes en los datos.

En ambos casos, el modelo _aprende_ algo en base a los datos de forma que después pueda _aplicar_ lo aprendido a datos nuevos, y determinar:
- Si el nuevo dato presenta algún patrón observado previamente
- Si el nuevo dato corresponde a algún grupo

Y en ambos casos, para determinar esto el modelo realiza **Operaciones matemáticas** con los atributos para obtener su respuesta de acuerdo a **reglas** determinadas por el **algoritmo del modelo**. El algoritmo define cómo encontrar los **parámetros** del modelo que permitirán operar los atributos para obtener el resultado que minimíce una **métrica**. Un pseudocódigo para entender esto podría ser:


```python
# Set de reglas en que se operan los atributos
import AlgoritmoMachineLearning 

# Opciones no aprendibles que definen variaciones al algoritmo
hiperparametros = {

}

# Modelo: instancia del algoritmo con los hiperparámetros el cual puede aprender de los datos
modelo = AlgoritmoMachineLearning(**hiperparametros)

print(modelo._parametros) # Imprime None. El modelo comienza en blanco

# Entrenar modelo con datos y objetivo a aprender (opcional)
if AlgoritmoMachineLearning.es_supervisado:
    modelo._parametros = modelo.fit(datos, columna_objetivo)
else:
    modelo._parametros = modelo.fit(datos)

# Modelo puede predecir lo aprendido con datos nuevos. Sus parámetros reflejan el aprendizaje
prediccion = modelo.predict(datos_nuevos, modelo._parametros)
```

## Aprendizaje Supervisado

En aprendizaje supervisado se basa en trabajar con datasets cuyas observaciones son conjuntos de vectores con distintas **características** que describen a algún objeto más una **etiqueta/valor real** la cuál asigna una clase o valor a cada objeto. 

Cuando el valor a predecir es un(a): 

- Categoría/Etiqueta, el problema que se resuelve se denomina **Clasificación**.
- Valor real, el problema que se resuelve se denomina **Regresión**.


Las etiquetas pertenecen a un número finito de clases.


Por ejemplo, el caso que estemos describiendo a una persona, el vector puede contener las siguentes **features/características**:

- su altura en cm, 
- edad, 
- peso en kg, 
- residencia, 
- etc...

Mientras que la **etiqueta** puede ser si la persona *quiere o no contratar un servicio de internet*: $\{ True, False\}$ 

In [1]:
import pandas as pd

# ejemplo clasificación
pd.DataFrame(
    [[177, 43, 72, "Maipú", True], [160, 16, 60, "Pudahuel", False]],
    columns=["Altura", "Edad", "Peso", "Residencia", "Posible cliente?"],
)

,Altura,Edad,Peso,Residencia,Posible cliente?
0,177,43,72,Maipú,True
1,160,16,60,Pudahuel,False


Como también lo que está dispuesto a gastar en el plan.

In [2]:
# ejemplo regresión
pd.DataFrame(
    [[177, 43, 72, "Maipú", 55000], [160, 16, 60, "Pudahuel", 0]],
    columns=["Altura", "Edad", "Peso", "Residencia", "Cuánto está dispuesto a pagar?"],
)

,Altura,Edad,Peso,Residencia,Cuánto está dispuesto a pagar?
0,177,43,72,Maipú,55000
1,160,16,60,Pudahuel,0


El objetivo final del aprendizaje supervisado es crear modelos que permiten **asignar de forma automática categorías o valores a observaciones nuevas**. 

En términos prácticos, dado una nueva observación representada por un vector de características, el modelo generado debe ser capaz de asignar una etiqueta a dicha observación.


### Un buen ajuste

El modelo construido debe **generalizar**, es decir, debe ser capaz de realizar predicciones correctas en nuevas observaciones. Para el ejemplo de clasificación, podemos considerar que el modelo generado está separando los datos por clases a través de un *límite de decisión*. Este límmite de decisión podría ser muy ajustado o más holgado, y muchas veces de esto depende qué tan bien puede generalizar

<div align='center'>
<img src='https://github.com/MDS7202/MDS7202/blob/main/recursos/2023-01/18-Aprendizaje-Supervisado-I/overfitting.png?raw=true' witdh=400/>

</div>

<div align='center'>
<a href='https://www.researchgate.net/figure/Example-of-overfitting-in-classification-a-Decision-boundary-that-best-fits-training_fig1_349186066'>Ejemplo de *Overfitting* en researchgate</a>
</div>

> **Pregunta ❓:** En la imágen, cuál de las 2 clasificación es mejor? Qué ventajas y desventajas tiene cada una?

#### Sobreajuste desde un enfoque generativo

Realizar machine learning implica considerar que la etiqueta buscada de alguna forma depende de los datos. Es decir, se asume el siguiente modelo:

```
variable_objetivo = modelo(datos)
```

o matemáticamente

$$Y=f(X)$$

Es decir, que la variable que se busca predecir es una función de los datos. Ciertamente, podríamos definir una función matemática que predice correctamente la etiqueta en base a los datos. Sin embargo, para entrenar un buen modelo debemos pensar en _por qué_ una variable objetivo podría predecirse en función de los datos? La respuesta es que existe un **fenómeno real** que produce relaciones entre diferentes variables, el cual buscamos modelar de alguna forma.

$$
\begin{align}
Z→X\\
Z→Y
\end{align}
$$

Donde $Z$ es el fenómeno real subyacente, $X$ son los datos observados e $Y$ es la variable a predecir. En esta relación, puede o no existir una relación causal entre $X$ e $Y$, pero sí existe un fenómeno subyacente que define ambos y por lo tanto permite definir $Y$ en función de $X$. Sin embargo, cabe notar 2 fenómenos:

- $X$ e $Y$ no necesariamente son **todas** las variables relevantes producidas por $Z$. Puede haber variables no observadas
- En el fenómeno $Z$, pueden existir procesos aleatoreos, por lo que incluso si se tuvieran todas las variables relevantes, existe incertidumbre y no es posible predecir de forma perfecta

Por esto, el modelo real que se busca obtener tiene en realidad la siguiente forma

$$Y = f(X) + \epsilon$$

Donde
- $X$ Son los datos disponibles
- $Y$ Es la etiqueta a predecir
- $f$ es el modelo predictivo
- $\epsilon$ es el error de predicción o incertidumbre

¿Qué tanto se puede acercar la predicción a la etiqueta real? Tanto como la relación real del **fenómeno subyacente** defina entre $X$ e $Y$. Si el modelo se ajusta de forma extremadamente precisa puede estar capturando no solo la relación real, sino también el ruido específico de esos datoses. En este caso, se habla que el modelo está **sobreajustado** ya que la relación entre $Y$ y $\epsilon$ no se verá reflejada en datos nuevos.


#### Ejemplo

Por ejemplo, en el caso de uso del **pronóstico del tiempo**, tenemos:
- $Z$ = estado atmosférico
  - Humedad
  - Presión
  - Temperatura
  - Dinámicas del aire

En este caso el fenómeno subyacente $Z$ implica que existen dinámicas generativas que producen que estas variables interactúen entre sí y se influencien, afectando sus valores en el futuro. En base a esto, podemos tener el siguiente problema de predicción.
- $X$ = datos disponibles
  - Densidad de nubes
  - Lecturas de humedad
  - Presión barométrica
  - Lecturas de termómetro
  - Fecha
  - Valor del dólar
  - Tasa de desempleo
  - Nivel de congestión de tráfico
  - Consumo elétrico
  - Nivel de actividad de twitter
- $Y$ = objetivo
  - Precipitaciones (mm) 

En este caso, variables como la densidad de nubes, humedad, presión y temperatura en días anteriores podrían ser un reflejo de fenómenos atmosféricos que influyen en la probabilidad de lluvia del futuro, y la fecha influye de qué forma estos influyen. Sin embargo, encontramos variables que _probablemente_ no influyen en la probabilidad de lluvia, como el valor del dólar, tasa de desempleo, etc... Si bien podría existir una relación que refleje un fenómeno subyacente (ej., mayor tráfico = mayor smog, consumo eléctrico depende del nivel de obscuridad, etc.) probablemente esta relación es muy débil, por lo que estas variables principalmente **ensucian** nuestro modelo. 

Adicionalmente, existen variables que no estamos observando y que serían útiles. Como las precipitaciones en días anteriores, presión y temperatura en lugares cercanos, fenómenos externos como la niña o el niño, etc. 

Toda esta información faltante es necesaria para tener un modelo confiable. Como no la tenemos, el modelo intenta predecir $Y$ en base al $X$ disponible, proceso en el cual puede capturar relaciones inexistentes: encontrar una relación entre el valor del dolar y la lluvia que en realidad es aleatorea, o buscar predecir exactamente la lluvia en base a lo que tiene. En ese caso el modelo estará **sobreajustado**.

Un modelo correctamente ajustado, predicirá todo lo de $Y$ que es realmente reflejado por $X$. Es decir, no reflejará una relación perfecta entre $X$ e $Y$, sino aquello que sí pueda encontrarse en datos nuevos.

> **Nota:** No existe forma en que el modelo 'sepa' qué tan bien ajustado está, o si la relación es o no extendible a datos nuevos. La única forma confiable de obtener esto es evaluar el desempeño del modelo siempre en datos no utilizados para entrenar, mientras se introducen **regularizaciones** en el algoritmo que se saben que reducen el sobreajuste de los modelos.

### Framework general de aprendizaje supervisado

Cuando realizamos aprendizaje supervisado, buscamos que en base a un **algoritmo**, un **modelo** aprenda las transformaciones matemáticas que permiten acercarse lo más posible a una etiqueta o valor deseado. Para lograr esto, se busca optimizar una **métrica**


#### Métricas


Las métricas son mediciones de qué tan bien se está desempeñando el modelo, o en otras palabras, qué tanto se parece la etiqueta predicha por el modelo de machine learning a la etiqueta real. Estas pueden ser métricas de **similitud** o métricas de **error**.

**Métricas de similitud**

Son métricas que indican *qué tanto se parece* la predicción del modelo a la etiqueta real. Mientras más alta la métrica, mejor el desempeño del modelo, por lo tanto se busca **maximizarlas** en el entrenamiento. Ejemplos:
- Clasificación: Accurracy, F1-score, ROC-AUC
- Regresión: $R^2$, 

**Métricas de error, o Loss**

Son métricas que indican *qué tanto se distancia* la predicción del modelo de la etiqueta real. Mientras más bajas, mejor el desempeño del modelo ya que indican que este tiene menos error, por lo tanto se busca **minimizarlas** en el entrenamiento. Ejemplos:
- Clasificación: Entropía cruzada
- Regresión: Error cuadrático medio (MSE), Error absoluto medio (MAE), RMSE

El _cómo_ minimizo o maximizo la métrica modificando los parámetros depende del algoritmo usado

#### Pipeline de aprendizaje supervisado

La siguiente lista muestra las etapas que debería cumplir un algoritmo de aprendizaje supervisado clásico (i.e., no red neuronal)

1. **Feature Engineering y Preprocesamiento**: Recolectar y preparar los datos.
2. **Entrenar** un algoritmo de clasificación/regresión usando los datos.
3. **Evaluar** qué tan bien clasifica el modelo generado.
4. **Optimizar los modelos** modificando sus hiperparámetros.

### Cómo determinar que algorimo utilizar 

Lo más importante es la **capacidad predictiva** del modelo.
Sin embargo, también hay otros factores muy relevantes que determinarán que algoritmo predictivo utilizar: 

**Eficiencia**: 
  - ¿Qué tanto se está demorando mi modelo en entrenar? 
  - ¿Y en predecir? 
  - ¿Es eficiente en memoria? 
  - ¿Debe almacenar el dataset de entrenamiento para funcionar?
  - ¿Es posible usarlo en tiempo real para algún tipo de solución online?
  
**Número de Features y Ejemplos Requeridos**: 
  - ¿Cuántos datos o features son requeridos para entrenar el modelo?
  - ¿Es compatible con la cantidad que dispongo?
  - ¿El tipo de features (i.e., categorícas, numéricas, combinación de ambas, etc...) es compatible con el algoritmo?
  
**Explicabilidad**: 
  - ¿Puedo explicar por qué el modelo está clasificando/regresionando de la manera que lo hace? 
  
***Fairness***: 
  - ¿Mi modelo es injusto con respecto a algún grupo social?

##  Matriz de Confusión

<div align='center'>
    <img src="https://github.com/MDS7202/MDS7202/blob/main/recursos/2023-01/18-Aprendizaje-Supervisado-I/matriz_conf.PNG?raw=true" alt="Ejemplo de una matriz de confusión para un problema de clasificación binario" width=450/>
</div>

<center>Ejemplo de una matriz de confusión para un problema de clasificación binario.</center>



---

> Ejemplo: Alergia a cierto medicamento en donde la clase `+` indica alergia.


Nuestro dataset tiene 10.000 observaciones distribuidos de la siguiente forma:

- Clase `+`: 100 observaciones.
- Clase `-`: 9900 observaciones.


Luego, creamos un modelo que clasificó nuestro dataset y graficamos sus resultados a través de la siguiente matriz de confusión:


|                    | **Predicha (`+`)**  | **Predicha (`-`)** |
|--------------------|---------------------|--------------------|
| **Real (`+`)**     | 10                  | 90                 |
| **Real (`-`)**     | 100                 | 9800               |

---

> **Pregunta**: ¿Cuales métricas de desempeño conocen para evaluar este caso?

### Métricas de desempeño

Métricas basadas en contar datos correcta e incorrectamente clasificados:

- **Accuracy (Exactitud)**: $$\text{accuracy} = \frac{\text{número de predicciones correctas}}{\text{número de predicciones totales}}$$

- **Error rate (Tasa de error)**: $$\text{error rate} = \frac{\text{número de predicciones incorrectas}}{\text{número de predicciones totales}}$$



- En nuestro ejemplo anterior: 

$$\text{accuracy} = \frac{9810}{10000} = 0.981$$

$$\text{error rate} = \frac{190}{10000} = 0.019$$


> **Pregunta ❓:** ¿Cuál es el problema de `Accuracy` en nuestro ejemplo?



---


#### Métricas Basadas en la Matriz de Confusión

Una posible solución a este problema son las métricas basadas en la matriz de confusión:

<div align='center'>
    <img src="https://github.com/MDS7202/MDS7202/blob/main/recursos/2023-01/18-Aprendizaje-Supervisado-I/matriz_conf.PNG?raw=true" alt="Ejemplo de una matriz de confusión para un problema de clasificación binario" width=450/>
</div>

- **Precision**:  Fracción de ejemplos correctamente predichos como `+` con respecto a todos los predichos `+`.

$$P = \frac{\text{Clasificados correctamente como positivo}}{\text{Todos los predichos como positivos}} =\frac{TP}{(TP + FP)}$$


<br>

- **Recall**: Fracción de ejemplos `+` que son correctamente clasificados: 

$$R = \frac{\text{Clasificados correctamente como positivo}}{\text{Todos los que debería haber clasificado como positivos}}  = \frac{TP}{(TP+FN)}$$


<br>

- **F1 measure**: Combina precisión y recall usando una media armónica (i.e., media que castiga si ambos valores son muy diferentes).

$$F = \frac{2PR}{(P+R)}$$



|                    | **Predicha (`+`)**  | **Predicha (`-`)** |
|--------------------|---------------------|--------------------|
| **Real (`+`)**     | 10                  | 90                 |
| **Real (`-`)**     | 100                 | 9800               |

En nuestro ejemplo anterior:


$$P = \frac{10}{110} = 0.\bar{09}$$

$$R = \frac{10}{100} = 0.1$$

$$F = \frac{2 \cdot 0.1 \cdot 0.\bar{1}}{(0.1 + 0.\bar{1})} \approx 0.095$$ 

Ahora claramente se nota el problema.

In [ ]:
p = 10 / 110
r = 10 / 100

f = 2 * p * r / (p + r)
f

0.09523809523809525

> **Pregunta ❓**: En el caso de un problema de detección de frutas podridas, ¿que nos convendría más utilizar? ¿precision o recall? 

![](https://www.lavanguardia.com/files/article_main_microformat/uploads/2019/11/05/5e9979d2bdba1.jpeg)

> **Pregunta ❓**: Para el caso de un trader, ¿que nos convendría más utilizar? ¿precision o recall?


![](https://www.eltiempo.com/files/article_main_1200/uploads/2023/03/02/640117d948243.jpeg)

#### Matriz de confusión multiclase

Cuando tenemos $k$ clases, la matriz de confusión es una matriz de $k \times k$.

<div align='center'>
<img src="https://github.com/MDS7202/MDS7202/blob/main/recursos/2023-01/18-Aprendizaje-Supervisado-I/matriz_conf_multiclase.png?raw=true" alt="Ejemplo de una matriz de confusión para un problema de clasificación binario" style="width: 500px;"/>
</div>

¿Cómo calculamos las métricas?


#### Métricas de Desempeño Generalizadas: 


- Precision: Fracción de ejemplos asignados a la clase `i` que son realmente de la clase `i`. Finalmente, es la fracción de casos en los que declaramos correctamente 𝑖 de todos los casos en los que el algoritmo declaró 𝑖.

$$\text{precision} = \frac{c_{ii}}{\sum_{j}c_{ji}}$$


- Recall: Fracción de ejemplos de la clase `i` correctamente clasificados. Recall es la fracción de eventos en los que declaramos correctamente 𝑖 de todos los casos en los que el verdadero estado 𝑖: 

$$\text{recall} = \frac{c_{ii}}{\sum_{j}c_{ij}}$$

- Accuracy: Fracción de ejemplos correctamente clasificados:

$$\text{accuracy} = \frac{\sum_{i}c_{ii}}{\sum_{j}\sum_{i}c_{ij}}$$

#### Estrategia de Agregación

- **Macroaveraging**
    - Computar métrica para cada clase y luego promediar. 
    - Sobrerepresentan clases minoritarias al tratar a todas por igual.

- **Weighted**
    - Computar métrica para cada clase y luego hace un promedio ponderado por el número de ejemplos de esa clase.
    - Al ser ponderado por el número de casos, da más prioridad a las clases frecuentes.


`Scikit` provee un acceso rápido a todas estas métricas a través de su función `sklearn.metrics.classification_report`

### Underfitting y Overfitting

Errores de entrenamiento o **Underfitting**. 
- Malos resultados sobre los datos de entrenamiento
- El clasificador no tiene capacidad de aprender el patrón.

Errores de generalización o **Overfitting**. 
- Malos resultados sobre datos nuevos 
- El modelo se hace demasiado específico a los datos de entrenamiento. 


<div align='center'>
<img src='https://github.com/MDS7202/MDS7202/blob/main/recursos/2023-01/18-Aprendizaje-Supervisado-I/tipos_fit.png?raw=true' width=800/>
</div>

<div align='center'>
    Fuente: The Hundred-Page Machine Learning Book.
</div>

> Nota: Una muy buena forma de revisar el sobre ajuste es a traves de las curvas de aprendizaje. Estas curvas permiten ver el sobreajuste a medida que se agregan nuevos datos.

![](https://i.stack.imgur.com/ViUnZ.png)